<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_02_Data_Analysis_with_Pandas_and_NumPy/Week_4_Data_Manipulation/Day_27_Advanced_Pandas_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 27: Advanced Pandas Techniques

## Phase 1 | Month 2 | Week 4 | Day 27

**Goal:** Learn advanced Pandas patterns used by senior data scientists — window functions, categorical dtypes, method chaining, and performance.

### What You Will Learn
1. `pd.Categorical` for memory-efficient string columns
2. Method chaining — writing Pandas like a pipeline
3. Window functions: `expanding()`, `rolling()`, `ewm()`
4. Pandas options and display settings
5. Memory optimisation for large DataFrames
6. Profiling pandas performance

### Resources
- 📖 [Effective Pandas — Tom Augspurger](https://tomaugspurger.github.io/posts/effective-pandas-1-bloated/)
- 🎥 [Pandas Best Practices — Kevin Markham](https://www.youtube.com/watch?v=dPwLlJkSHLo)
- 📖 [Modern Pandas (7-part series)](https://tomaugspurger.github.io/posts/modern-1-intro/)

In [ ]:
import pandas as pd
import numpy as np
sys_import = __import__('sys')
np.random.seed(42)

# CATEGORICAL dtype — huge memory savings for string columns
n = 500_000
df_object = pd.DataFrame({
    'county': np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'], n),
    'dept'  : np.random.choice(['IT','Finance','Health','Education','Infrastructure'], n),
    'salary': np.random.randint(40_000, 400_000, n),
})

df_cat = df_object.copy()
df_cat['county'] = df_cat['county'].astype('category')
df_cat['dept']   = df_cat['dept'].astype('category')

obj_mem = df_object.memory_usage(deep=True).sum() / 1024**2
cat_mem = df_cat.memory_usage(deep=True).sum() / 1024**2
print(f'DataFrame with {n:,} rows:')
print(f'  object dtype : {obj_mem:.1f} MB')
print(f'  category dtype: {cat_mem:.1f} MB')
print(f'  Memory savings: {(1 - cat_mem/obj_mem)*100:.0f}%')
print(f'\nCategory values (county): {df_cat["county"].cat.categories.tolist()}')

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# METHOD CHAINING — readable, maintainable pipelines
np.random.seed(0)
raw = pd.DataFrame({
    'name': [f'Employee_{i:03d}' for i in range(100)],
    'dept': np.random.choice(['IT','HR','Finance',None],100),
    'salary': np.random.choice([np.nan if np.random.rand()<0.1 else np.random.randint(40000,350000) for _ in range(100)]),
    'county': np.random.choice(['Nairobi','Mombasa','Kisumu'],100),
    'years': np.random.randint(0,30,100),
})

# The 'wrong' way: lots of intermediate variables
df1 = raw.dropna(subset=['dept','salary'])
df2 = df1[df1['salary'] > 50_000]
df3 = df2.copy()
df3['salary_bucket'] = pd.cut(df3['salary'], bins=[0,100000,200000,float('inf')], labels=['Junior','Senior','Executive'])
df4 = df3.groupby(['dept','salary_bucket'])['salary'].mean().reset_index()

# The 'right' way: method chaining
result = (
    raw
    .dropna(subset=['dept','salary'])
    .query('salary > 50_000')
    .assign(salary_bucket=lambda d: pd.cut(d['salary'], bins=[0,100000,200000,float('inf')], labels=['Junior','Senior','Executive']))
    .groupby(['dept','salary_bucket'])['salary']
    .mean()
    .reset_index()
    .rename(columns={'salary':'mean_salary'})
    .sort_values('mean_salary', ascending=False)
)
print('Method-chained result:')
print(result.round(0).head(8))

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# WINDOW FUNCTIONS
prices = pd.Series(
    np.round(50 + np.cumsum(np.random.normal(0,1,100)), 2),
    name='price'
)

df = pd.DataFrame({'price': prices})
df['rolling_mean_7'] = df['price'].rolling(7).mean()
df['rolling_std_7']  = df['price'].rolling(7).std()
df['expanding_mean'] = df['price'].expanding().mean()  # running mean
df['ewm_alpha0.3']   = df['price'].ewm(alpha=0.3).mean()

print('Window functions (last 5 rows):')
print(df.tail(5).round(3))

# Lag and lead features (important for ML on time series!)
df['lag_1']  = df['price'].shift(1)   # yesterday's price
df['lag_7']  = df['price'].shift(7)   # week ago
df['lead_1'] = df['price'].shift(-1)  # tomorrow's price (target!)
df['return'] = df['price'].pct_change()
print('\nLag/lead features (rows 10-15):')
print(df[['price','lag_1','lag_7','lead_1','return']].iloc[10:16].round(4))

## Exercise: Memory Optimisation Challenge

Optimise a large DataFrame to use minimum memory.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

n = 200_000
df = pd.DataFrame({
    'id'        : np.arange(1, n+1, dtype=np.int64),
    'county'    : np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'], n),
    'dept'      : np.random.choice(['IT','HR','Finance','Education','Health'], n),
    'score'     : np.random.uniform(0, 100, n).astype(np.float64),
    'flag'      : np.random.choice([0, 1], n).astype(np.int64),
    'age'       : np.random.randint(18, 65, n).astype(np.int64),
})

print(f'Original memory: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print(df.dtypes)

# Optimise:
# 1. Convert 'county', 'dept' to category
# 2. Downcast 'score' to float32
# 3. Downcast 'flag' to bool or int8
# 4. Downcast 'age' to uint8
# 5. Downcast 'id' to int32
# Report final memory usage and % savings
# YOUR CODE HERE

## Summary

- `category` dtype saves 50-90% memory for string columns
- Method chaining produces readable, maintainable pipelines
- `expanding()`: cumulative statistics; `rolling()`: sliding window
- Lag features are essential for time series ML
- Always downcast numeric types to reduce memory footprint

**Next: Day 28 — Month 2 Final Review and Capstone Project**